# 13 — Attention and a tiny causal transformer

**Level:** Advanced · **Study time:** 90–120 minutes (including experiments)

**Prerequisites:** Notebooks 02, 07, and 11–12.

## What you will be able to do
- Compute scaled dot-product attention by hand.
- Apply a causal mask and verify no future information leaks.
- Train and sample from a tiny next-token model.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 12](12_sequence_forecasting.ipynb) · [Next: 14](14_autoencoders.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Inspect attention weights

A query is compared with keys to decide how to mix values. Scores are divided by the square root of key width before softmax. A causal mask blocks future positions. Attention weights describe this mixing operation, not a complete explanation of model decisions.


In [ ]:
import math
Q, K, V = torch.randn(1, 4, 8), torch.randn(1, 4, 8), torch.randn(1, 4, 8)
scores = Q @ K.transpose(-2, -1) / math.sqrt(Q.shape[-1])
blocked = torch.triu(torch.ones(4, 4, dtype=torch.bool), diagonal=1)
weights = scores.masked_fill(blocked, float('-inf')).softmax(-1)
attended = weights @ V
assert torch.allclose(weights.sum(-1), torch.ones(1, 4))
assert torch.equal(weights[0][blocked], torch.zeros(6))
plt.imshow(weights[0], vmin=0, vmax=1, cmap="Blues")
plt.xlabel("Key position"); plt.ylabel("Query position"); plt.colorbar(); plt.show()


## Step 3 — Create a miniature next-token task

Our vocabulary is the digits 0 through 7. Each sequence follows a cyclic rule, starting at a random digit. Inputs are all tokens except the last; targets are shifted one position forward. This controlled task makes masking and optimization easy to verify, but is much simpler than natural language.


In [ ]:
vocab_size, context_length = 8, 12
def make_sequences(n, generator):
    start = torch.randint(vocab_size, (n, 1), generator=generator)
    return (start + torch.arange(context_length + 1)) % vocab_size
generator = torch.Generator().manual_seed(SEED)
sequences = make_sequences(320, generator)
train_seq, val_seq = sequences[:256], sequences[256:]
loader = DataLoader(TensorDataset(train_seq[:, :-1], train_seq[:, 1:]), batch_size=32, shuffle=True)
print("Input:", train_seq[0, :-1].tolist(), "Target:", train_seq[0, 1:].tolist())


## Step 4 — Build one transformer block

Token embeddings encode identity; position embeddings encode order. Multiple attention heads learn separate mixing patterns. Residual additions preserve a direct information path, LayerNorm stabilizes scales, and a feed-forward network transforms each position. Here `MultiheadAttention` boolean True means **blocked**; mask conventions differ across APIs.


In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self, width=32):
        super().__init__()
        self.token = nn.Embedding(vocab_size, width)
        self.position = nn.Embedding(context_length, width)
        self.norm1, self.norm2 = nn.LayerNorm(width), nn.LayerNorm(width)
        self.attention = nn.MultiheadAttention(width, num_heads=4, dropout=0., batch_first=True)
        self.ff = nn.Sequential(nn.Linear(width, width * 2), nn.GELU(), nn.Linear(width * 2, width))
        self.output = nn.Linear(width, vocab_size)
    def forward(self, tokens):
        length = tokens.shape[1]
        x = self.token(tokens) + self.position(torch.arange(length, device=tokens.device))
        mask = torch.triu(torch.ones(length, length, device=tokens.device, dtype=torch.bool), diagonal=1)
        z = self.norm1(x)
        mixed, _ = self.attention(z, z, z, attn_mask=mask, need_weights=False)
        x = x + mixed
        x = x + self.ff(self.norm2(x))
        return self.output(x)

model = TinyTransformer()
assert model(train_seq[:2, :-1]).shape == (2, context_length, vocab_size)


## Step 5 — Train and verify causality

Cross entropy needs one class dimension, so flatten batch and time together. Causality is a property we can test: changing a future token must leave earlier logits unchanged in evaluation mode.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=0.01)
for epoch in range(12):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = nn.functional.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
        loss.backward(); optimizer.step()
model.eval()
with torch.no_grad():
    logits = model(val_seq[:, :-1])
    accuracy = (logits.argmax(-1) == val_seq[:, 1:]).float().mean().item()
    original = val_seq[:1, :-1].clone()
    changed = original.clone(); changed[:, -1] = (changed[:, -1] + 3) % vocab_size
    assert torch.allclose(model(original)[:, :-1], model(changed)[:, :-1], atol=1e-5)
print("Validation token accuracy on the cyclic task:", accuracy)
assert accuracy > 0.95


## Step 6 — Generate autoregressively

At inference time the next true token is unavailable. Predict one token, append it to the context, then repeat. Greedy decoding takes the highest-scoring token; sampling introduces controlled randomness.


In [ ]:
generated = torch.tensor([[3]])
with torch.no_grad():
    for _ in range(18):
        context = generated[:, -context_length:]
        next_token = model(context)[:, -1].argmax(-1, keepdim=True)
        generated = torch.cat([generated, next_token], dim=1)
print("Generated:", generated[0].tolist())


## Your turn — practice before opening the solutions

1. Compare manual attention with `scaled_dot_product_attention(..., is_causal=True)`.
2. Count this transformer's trainable parameters.
3. Sample a next token at temperatures 0.5 and 2.0. Explain how temperature affects probabilities.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
fast_attention = nn.functional.scaled_dot_product_attention(Q, K, V, is_causal=True, dropout_p=0.)
assert torch.allclose(attended, fast_attention, atol=1e-6)
print("Parameters:", sum(p.numel() for p in model.parameters()))
with torch.no_grad():
    last_logits = model(torch.tensor([[3, 4]]))[:, -1]
    for temperature in [0.5, 2.0]:
        distribution = (last_logits / temperature).softmax(-1)
        print(temperature, distribution.round(decimals=3), torch.multinomial(distribution, 1).item())
# Higher temperature flattens the distribution; lower temperature sharpens it.


## Before moving on

This notebook teaches transformer mechanics. Expert language-model work also requires realistic corpora, tokenizer design, stronger evaluation, positional strategies, and memory-aware training. The cyclic validation task contains repeated patterns and does not demonstrate language understanding.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [MultiheadAttention](https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html)
- [Scaled dot-product attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)


[Course guide](../README.md) · [Previous: 12](12_sequence_forecasting.ipynb) · [Next: 14](14_autoencoders.ipynb)
